# Prerequisites 

### Databricks Serverless + Unity Catalog + AWS S3 Setup

#### Step 1 — Create S3 Bucket
- Go to AWS S3 Console.
- Create a new bucket (e.g., `rj-de-bucket`).
- Upload your CSV, Delta, and Parquet files as needed.

#### Step 2 — Create IAM Policy
- Go to **AWS Console → IAM → Policies → Create Policy**.
- Paste the following policy (replace bucket name if needed):

json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "s3:ListBucket",
        "s3:GetBucketLocation",
        "s3:GetBucketNotification",
        "s3:PutBucketNotification"
      ],
      "Resource": "arn:aws:s3:::bucket-name"
    },
    {
      "Effect": "Allow",
      "Action": [
        "s3:GetObject",
        "s3:PutObject",
        "s3:DeleteObject"
      ],
      "Resource": "arn:aws:s3:::bucket-name/*"
    }
  ]
}

- Save policy (e.g., `databricks-s3-policy`).

#### Step 3 — Create IAM Role
- Go to **IAM → Roles → Create Role**.
- Choose **AWS Account**.
- Attach the policy `databricks-s3-policy`.
- Create role (e.g., `databricks-s3-role`).

#### Step 4 — Create Storage Credential in Databricks
- Go to **Databricks → Catalog → External Data → Credentials → Create Credential**.
- Fill:
  - Credential Name: `databricks-s3-credentials`
  - Credential Type: IAM Role
  - IAM Role ARN: (your role ARN, e.g., `arn:aws:iam::<account-id>:role/databricks-s3-role`)
- Click **Create**.

#### Step 5 — Copy External ID
- After creation, copy the **External ID** 

#### Step 6 — Configure Trust Relationship
- Go to **IAM → Roles → databricks-s3-role → Trust Relationships → Edit Trust Policy**.
- Replace with:

json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Principal": {
        "AWS": "arn:aws:iam::<account-id>:root"
      },
      "Action": "sts:AssumeRole",
      "Condition": {
        "StringEquals": {
          "sts:ExternalId": "your-external-id"
        }
      }
    }
  ]
}

- Replace the **External ID** with yours.
- Save policy.

#### Step 7 — Validate Credential
- In Databricks: **Credentials → databricks-s3-credentials → Validate Configuration**.
- Ensure all permissions show as ✅ (Read, Write, List, Delete, AssumeRole).

#### Step 8 — Create External Location
- Go to **Catalog → External Data → External Locations → Create External Location**.
- Fill:
  - Name: `s3_location`
  - URL: `s3://bucket-name/`
  - Storage Credential: `databricks-s3-credentials`

## DELTA TABLES

### Convert Employees.csv in workspace to Delta Table

In [0]:
df = spark.read.csv("/Workspace/Users/202301040298@mitaoe.ac.in/Employees.csv", header=True, inferSchema=True)



In [0]:
df.write.format("delta").mode("overwrite").option("header","true").save("s3a://rj-de-bucket/employees-data/")
print("Data written to S3 is succesfully")

Data written to S3 is succesfully


In [0]:
df_read_s3 = spark.read.format("delta").load("s3a://rj-de-bucket/employees-data/")
print("Data reading from S3 as delta table is succesfully")

Data reading from S3 as delta table is succesfully


In [0]:
display(df_read_s3)

EmpID,Name,Department,City,Gender,Salary,Experience,Project
1001,Amit1,IT,Pune,M,35000,1,Alpha
1002,Priya2,HR,Delhi,F,40000,2,Delta
1003,Raj3,Finance,Hyderabad,M,45000,3,Beta
1004,Neha4,Sales,Mumbai,F,50000,4,Omega
1005,Karan5,Marketing,Bengaluru,M,55000,5,Gamma
1006,Sneha6,IT,Pune,F,60000,6,Alpha
1007,Rahul7,HR,Delhi,M,65000,7,Delta
1008,Pooja8,Finance,Hyderabad,F,70000,8,Beta
1009,Vivek9,Sales,Mumbai,M,75000,9,Omega
1010,Anjali10,Marketing,Bengaluru,F,80000,10,Gamma


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# First, get the schema from existing Delta table
existing_df = spark.read.format("delta").load("s3a://rj-de-bucket/employees-data/")
print("Existing schema:")
existing_df.printSchema()

# CREATE: Insert new employee records with matching schema
new_data = [
    (2001, "Aryan", "IT", "Pune", "M", 75000, 3, "Sigma"),
    (2002, "Sanya", "Finance", "Mumbai", "F", 80000, 4, "Theta")
]

# Create DataFrame with the same schema as existing table
new_df = spark.createDataFrame(new_data, schema=existing_df.schema)

# Append to Delta table
new_df.write.format("delta").mode("append").save("s3a://rj-de-bucket/employees-data/")

print("✓ New employees inserted successfully")

Existing schema:
root
 |-- EmpID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Salary: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- Project: string (nullable = true)

✓ New employees inserted successfully


## Delta Tables

In [0]:
from delta.tables import DeltaTable

In [0]:
df_1 = spark.read.format("delta").load("s3a://rj-de-bucket/employees-data/")
display(df_1)

EmpID,Name,Department,City,Gender,Salary,Experience,Project
1001,Amit1,IT,Pune,M,35000,1,Alpha
1002,Priya2,HR,Delhi,F,40000,2,Delta
1003,Raj3,Finance,Hyderabad,M,45000,3,Beta
1004,Neha4,Sales,Mumbai,F,50000,4,Omega
1005,Karan5,Marketing,Bengaluru,M,55000,5,Gamma
1006,Sneha6,IT,Pune,F,60000,6,Alpha
1007,Rahul7,HR,Delhi,M,65000,7,Delta
1008,Pooja8,Finance,Hyderabad,F,70000,8,Beta
1009,Vivek9,Sales,Mumbai,M,75000,9,Omega
1010,Anjali10,Marketing,Bengaluru,F,80000,10,Gamma


## UPSERT

## Step 1 : Create new DataFrame 

In [0]:


# Read existing data
df_existing = spark.read.format("delta").load("s3a://rj-de-bucket/employees-data/")



data = [(10000, "Rohan", "HR", "Delhi", "M", 90000, 5, "Omega"),(1002, "Priya2", "HR", "Delhi", "F", 40000,2,"Delta")]

df_2 = spark.createDataFrame(data, schema=df_existing.schema)

display(df_2)

EmpID,Name,Department,City,Gender,Salary,Experience,Project
10000,Rohan,HR,Delhi,M,90000,5,Omega
1002,Priya2,HR,Delhi,F,40000,2,Delta


## Step 2 : Create Delta Table Object & Perform UPSERT

In [0]:


from delta.tables import DeltaTable
dlt_obj = DeltaTable.forPath(spark, "s3a://rj-de-bucket/employees-data/")


dlt_obj.alias("target").merge(df_2.alias("updates"), "target.EmpID = updates.EmpID").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## Step 3 : Read Final Data

In [0]:
df_final = spark.read.format("delta").load("s3a://rj-de-bucket/employees-data/")
display(df_final)

EmpID,Name,Department,City,Gender,Salary,Experience,Project
1001,Amit1,IT,Pune,M,35000,1,Alpha
1003,Raj3,Finance,Hyderabad,M,45000,3,Beta
1004,Neha4,Sales,Mumbai,F,50000,4,Omega
1005,Karan5,Marketing,Bengaluru,M,55000,5,Gamma
1006,Sneha6,IT,Pune,F,60000,6,Alpha
1007,Rahul7,HR,Delhi,M,65000,7,Delta
1008,Pooja8,Finance,Hyderabad,F,70000,8,Beta
1009,Vivek9,Sales,Mumbai,M,75000,9,Omega
1010,Anjali10,Marketing,Bengaluru,F,80000,10,Gamma
1011,Rohit11,IT,Pune,M,85000,1,Alpha
